# 02 — Model experiments

Compares the logged experiments (guide §19). All numbers are read from `reports/metrics/`, which is written by `make train`, `make tune`, `make evaluate`, `make transformer` and `make domain-eval`; nothing here is typed in by hand.

- **fs1** = the guide's word TF-IDF; **fs2** = word TF-IDF with emoji/punctuation tokens + character n-grams; **transformer** = DistilBERT (E4).
- E1–E4 are scored on **validation**. `E3-final` / `E4-final` use the test set once per model version.
- The Mentor-domain eval set (`data/mentor_eval/`) measures what matters for the app.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from src import config

experiments = pd.read_csv(config.EXPERIMENTS_CSV)
experiments["run"] = experiments["id"] + " / " + experiments["features"]
experiments

## Headline metrics per experiment

In [ ]:
latest = experiments.drop_duplicates(subset=["run", "eval_split"], keep="last").set_index("run")
metric_cols = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]

fig, ax = plt.subplots(figsize=(12, 5))
latest[metric_cols].plot.bar(ax=ax, rot=30)
ax.set_ylim(0, 1)
ax.set_title("Experiment comparison (validation, except *-final = test)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(config.FIGURES / "experiment_comparison.png", dpi=200)
latest[["model", "eval_split", *metric_cols, "cv_macro_f1"]]

## Per-class F1 on validation

Macro F1 hides class imbalance; this table shows which Mentor classes are hard.

In [ ]:
per_class = {}
for path in sorted(config.METRICS.glob("E*_validation.json")):
    report = json.loads(path.read_text())
    key = f"{report['id']} / {report.get('features', 'fs1')}"
    per_class[key] = {
        label: stats["f1-score"]
        for label, stats in report["metrics"]["per_class"].items()
        if isinstance(stats, dict) and "support" in stats and label not in {"macro avg", "weighted avg", "micro avg"}
    }
per_class = pd.DataFrame(per_class)

fig, ax = plt.subplots(figsize=(10, 5))
per_class.plot.barh(ax=ax)
ax.set_xlim(0, 1)
ax.set_title("Per-class F1 (validation)")
fig.tight_layout()
fig.savefig(config.FIGURES / "per_class_f1_validation.png", dpi=200)
per_class.round(4)

## Mentor-domain evaluation set

154 check-in style sentences (`data/mentor_eval/mentor_eval.csv`). This is the acceptance test for the app; GoEmotions scores do not measure it.

In [ ]:
domain = {}
for path in sorted(config.METRICS.glob("domain_eval_*.json")):
    r = json.loads(path.read_text())
    domain[r["model_version"]] = {
        "macro_f1": r["metrics"]["macro_f1"],
        "accuracy": r["metrics"]["accuracy"],
        **{f"acc_{k}": v["accuracy"] for k, v in r["accuracy_by_kind"].items()},
        "labels_reviewed": r["labels_reviewed"],
    }
domain = pd.DataFrame(domain).T
if len(domain):
    fig, ax = plt.subplots(figsize=(9, 4))
    domain[["macro_f1", "accuracy"]].astype(float).plot.bar(ax=ax, rot=15)
    ax.set_ylim(0, 1)
    ax.set_title("Mentor-domain eval set")
    fig.tight_layout()
    fig.savefig(config.FIGURES / "domain_eval_comparison.png", dpi=200)
domain

## Hyperparameter search (cross-validation on train only)

In [ ]:
for path in sorted(config.METRICS.glob("E3_*cv_results.csv")):
    print(path.name)
    display(pd.read_csv(path).head(8))

## Confusion matrices (row-normalized)

In [ ]:
for path in sorted(config.FIGURES.glob("confusion_matrix_*_normalized.png")):
    print(path.name)
    display(Image(filename=str(path), width=560))